# ⚠️ Google Colab Setup (Read First)
To run this lab successfully, you **must** enable the Free GPU.

1. In the menu bar at the top, click **Runtime** > **Change runtime type**.
2. Under "Hardware accelerator", select **T4 GPU** and click **Save**.
3. *(Note: You must be signed in with a free Google account to use Colab)*

> **💡 Solution Available:** Try completing this lab on your own first. When you're done, open **`C1_HOL3_AutoModel_and_Manual_Inference_Solution.ipynb`** from the file browser (left panel) in the same folder to compare your outputs with the expected exemplar.


# C1-HOL3: Load, Inspect, and Run Manual Inference Across Configurations

| Field | Detail |
|-------|--------|
| **Course** | Getting Started with Hugging Face Transformers |
| **Module** | M3: Loading and Running Models with AutoModel |
| **Complexity** | Medium |
| **Duration** | 18 minutes |
| **Environment** | Google Colab free T4 GPU |

---

## Learning Objective

**LO3 (Apply):** Load pre-trained models using the appropriate AutoModel class, inspect model configuration, run manual inference, and load models in reduced precision with `device_map="auto"`.

---

## Scenario

It is **Wednesday afternoon at NovaPay**. Sara's feasibility study is progressing, and now the ML team lead **Arun** is asking:

> *"Before we commit to deploying this model, I need to understand the infrastructure requirements. How much memory does it actually need? Can we run it in reduced precision to cut costs?"*

You take manual control of inference to answer these questions.

---

## Setup

Run the cell below to install required packages and verify GPU availability.

In [ ]:
# Setup — install required packages
!pip install -q transformers torch accelerate
# Imports
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    AutoModel
)
import warnings
warnings.filterwarnings('ignore')

# Verify GPU
if torch.cuda.is_available():
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU detected. Some precision comparisons will be limited.")
    print("   For the full experience, use Google Colab with a T4 GPU.")

print("\n✅ Setup complete!")

In [ ]:
# Sample NovaPay customer messages (reused from C1-HOL1)
messages = [
    "My payment was declined and I need help immediately!",
    "Love the new app design, very intuitive and clean.",
    "I've been waiting 3 days for my refund. This is unacceptable.",
    "Can you explain the fee structure for international transfers?",
    "Your customer service team was incredibly helpful today, thank you!"
]

checkpoint = "distilbert-base-uncased-finetuned-sst-2-english"

---

## Task 1: Load with the Right AutoModel Class (4 min)

Load a text classification checkpoint using `AutoModelForSequenceClassification`. Inspect the model config. Then attempt to load the same checkpoint with generic `AutoModel` and observe the difference.

In [ ]:
# Load with the CORRECT class — AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(checkpoint)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

# TODO: Inspect the model configuration
# Print: number of labels, label-to-id mapping, architecture name
print("=== Model Configuration (Correct Class) ===")
print(f"  Architecture:    {model.config.architectures}")
print(f"  Number of labels: {model.config.num_labels}")
print(f"  Label mapping:    {model.config.id2label}")
print(f"  Hidden size:      {model.config.hidden_size}")
print(f"  Total parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# TODO: Load the SAME checkpoint with generic AutoModel
# Observe what's different — the classification head is missing!
base_model = AutoModel.from_pretrained(checkpoint)

print("=== Model Comparison ===")
print(f"\n  AutoModelForSequenceClassification:")
print(f"    Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"    Has classifier head: Yes")
print(f"    Output type: logits (num_labels={model.config.num_labels})")

print(f"\n  AutoModel (generic):")
print(f"    Parameters: {sum(p.numel() for p in base_model.parameters()):,}")
print(f"    Has classifier head: No")
print(f"    Output type: hidden states (hidden_size={base_model.config.hidden_size})")

param_diff = sum(p.numel() for p in model.parameters()) - sum(p.numel() for p in base_model.parameters())
print(f"\n  Difference: {param_diff:,} parameters (the classification head)")
print(f"\n💡 Using the wrong AutoModel class means no task-specific output layer!")

#### ✅ Verification
- The `AutoModelForSequenceClassification` version should have **more parameters** (extra classification head)
- The `id2label` mapping should show `{0: 'NEGATIVE', 1: 'POSITIVE'}` or similar
- The `AutoModel` version should have fewer parameters and no classification output

---

## Task 2: Run Manual Inference (4 min)

Tokenize a batch of NovaPay messages, feed them directly into the model, extract logits, apply softmax, and map predictions to label names. Compare with the `pipeline()` output from C1-HOL1.

In [ ]:
# TODO: Tokenize the messages manually
# Use return_tensors="pt", padding=True, truncation=True
inputs = tokenizer(messages, return_tensors="pt", padding=True, truncation=True)

print(f"Input IDs shape:      {inputs['input_ids'].shape}")
print(f"Attention mask shape: {inputs['attention_mask'].shape}")

In [ ]:
# TODO: Run manual inference
# 1. Set model to eval mode
# 2. Use torch.no_grad() for inference
# 3. Pass inputs to model
# 4. Apply softmax to logits

model.eval()
with torch.no_grad():
    outputs = model(**inputs)

# Extract logits and apply softmax
logits = outputs.logits
probs = torch.softmax(logits, dim=-1)

# Map predictions to labels
print("=== Manual Inference Results ===")
for i, msg in enumerate(messages):
    pred_id = torch.argmax(probs[i]).item()
    pred_label = model.config.id2label[pred_id]
    pred_score = probs[i][pred_id].item()
    print(f"\n  Message: {msg[:50]}...")
    print(f"  Label:   {pred_label}")
    print(f"  Score:   {pred_score:.4f}")
    print(f"  Logits:  {logits[i].tolist()}")

In [ ]:
# Verify: Compare manual results with pipeline results
from transformers import pipeline as hf_pipeline

pipe = hf_pipeline("text-classification", model=checkpoint)
pipe_results = pipe(messages)

print("=== Pipeline vs Manual Comparison ===")
for i, msg in enumerate(messages):
    manual_id = torch.argmax(probs[i]).item()
    manual_label = model.config.id2label[manual_id]
    manual_score = probs[i][manual_id].item()
    
    pipe_label = pipe_results[i]['label']
    pipe_score = pipe_results[i]['score']
    
    match = "✅" if manual_label == pipe_label else "❌"
    print(f"  {match} Manual: {manual_label} ({manual_score:.4f}) | Pipeline: {pipe_label} ({pipe_score:.4f})")

#### ✅ Verification
All manual inference results should **exactly match** the pipeline results. Both the label and score should be identical (or within floating-point tolerance).

---

## Task 3: Compare Precision and Memory (6 min)

Load the same model in **float32** and **bfloat16**. Compare memory footprints. Then load a model with `device_map="auto"` to see automatic device placement.

> **Note:** For DistilBERT-sized models (~260MB), the precision savings are small. In production with billion-parameter models, the savings are dramatic (e.g., 14GB → 7GB for a 7B model).

In [ ]:
# Clean up previous models to free memory
del model, base_model, pipe
if torch.cuda.is_available():
    torch.cuda.empty_cache()

# TODO: Load the model in float32 (default)
model_fp32 = AutoModelForSequenceClassification.from_pretrained(checkpoint)
fp32_memory = model_fp32.get_memory_footprint() / 1e6
print(f"FP32 memory: {fp32_memory:.1f} MB")
print(f"FP32 dtype:  {next(model_fp32.parameters()).dtype}")

In [ ]:
# TODO: Load the model in bfloat16
# Hint: Use torch_dtype=torch.bfloat16 parameter
model_bf16 = AutoModelForSequenceClassification.from_pretrained(
    checkpoint, torch_dtype=torch.bfloat16
)
bf16_memory = model_bf16.get_memory_footprint() / 1e6
print(f"BF16 memory: {bf16_memory:.1f} MB")
print(f"BF16 dtype:  {next(model_bf16.parameters()).dtype}")

# Compare
savings = fp32_memory - bf16_memory
savings_pct = (savings / fp32_memory) * 100
print(f"\n=== Memory Comparison ===")
print(f"  FP32:    {fp32_memory:.1f} MB")
print(f"  BF16:    {bf16_memory:.1f} MB")
print(f"  Savings: {savings:.1f} MB ({savings_pct:.0f}%)")
print(f"\n💡 For a 7B parameter model, this would save ~7 GB of memory!")

In [ ]:
# TODO: Check if bfloat16 affects output quality
# Run inference with both models and compare
test_input = tokenizer(messages[0], return_tensors="pt")

model_fp32.eval()
model_bf16.eval()

with torch.no_grad():
    out_fp32 = model_fp32(**test_input)
    out_bf16 = model_bf16(**test_input)

probs_fp32 = torch.softmax(out_fp32.logits, dim=-1)
probs_bf16 = torch.softmax(out_bf16.logits.float(), dim=-1)  # Cast to float for comparison

print("=== Output Quality Comparison ===")
print(f"  FP32 probabilities: {probs_fp32[0].tolist()}")
print(f"  BF16 probabilities: {probs_bf16[0].tolist()}")
print(f"  Max difference:     {torch.max(torch.abs(probs_fp32 - probs_bf16)).item():.6f}")
print(f"  Same prediction:    {torch.argmax(probs_fp32) == torch.argmax(probs_bf16)}")

In [ ]:
# Load with device_map="auto" for automatic device placement
# This is most useful for large models that span multiple GPUs
del model_fp32, model_bf16
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model_auto = AutoModelForSequenceClassification.from_pretrained(
    checkpoint,
    device_map="auto",
    torch_dtype=torch.bfloat16
)

print(f"=== device_map='auto' ===")
print(f"  Model device: {model_auto.device}")
print(f"  Memory: {model_auto.get_memory_footprint() / 1e6:.1f} MB")
print(f"\n💡 device_map='auto' automatically places model layers on available devices.")
print(f"   For multi-GPU setups, it can split a model across GPUs.")

#### ✅ Verification
- BF16 memory should be approximately **half** of FP32 memory
- Predictions should be the **same** (or very close) between FP32 and BF16
- `device_map="auto"` should place the model on GPU (if available)

---

## Task 4: Complete the Hardware Planning Note (4 min)

### 📋 Deliverable D1: Hardware Planning Note for Arun

| Question | Answer |
|----------|--------|
| **Which AutoModel class is correct for this task?** | *TODO: Class name and why* |
| **FP32 memory footprint** | *TODO: MB* |
| **BF16 memory footprint** | *TODO: MB* |
| **Memory savings from BF16** | *TODO: MB and %* |
| **Does BF16 affect output quality?** | *TODO: Yes/No — what did you observe?* |
| **Recommended precision for deployment** | *TODO: FP32 or BF16?* |
| **GPU required?** | *TODO: Yes/No — and why?* |

### Infrastructure Recommendation

*TODO: Write 3-4 sentences recommending the infrastructure setup for deploying this model at NovaPay. Consider: precision, GPU vs CPU, expected latency, and scaling.*

---

## Key Takeaways

In this lab, you:
1. **Chose the correct AutoModel class** — `AutoModelForSequenceClassification` includes the task-specific head
2. **Ran manual inference** — tokenize → forward pass → softmax → label mapping
3. **Compared precision formats** — BF16 cuts memory ~50% with negligible quality loss
4. **Used `device_map="auto"`** — automatic device placement for efficient GPU utilization
5. **Wrote infrastructure recommendations** — translating technical findings into business decisions

**Next up:** Thursday (C1-HOL4) — you'll build the final model evaluation report for Sara, integrating everything from Modules 1-4.

---
*© NovaPay Feasibility Study — Day 3 (Wednesday afternoon)*